In [ ]:
# =============================================================================
# Zelle 01 – Projektpfad bestimmen und Ordnerstruktur anlegen
# =============================================================================
# Zweck: Die gesamte Ordnerstruktur wird per Skript erzeugt statt von Hand.
#   Damit ist sie reproduzierbar (z. B. auf dem Medion) und dokumentiert.
# Idempotent: Mehrfaches Ausführen ist unschädlich, vorhandene Ordner
#   bleiben unverändert (exist_ok=True).
# .gitkeep: Git versioniert keine leeren Ordner. Eine leere Platzhalter-Datei
#   sorgt dafür, dass die Struktur später im Repo sichtbar ist.
# =============================================================================

from pathlib import Path
import csv
import sys

# --- Projektwurzel bestimmen -------------------------------------------------
# VS Code startet ein Notebook im Ordner des Notebooks (hier: notebooks/).
# Die Projektwurzel liegt dann eine Ebene darüber. Nicht hart codiert, damit
# derselbe Code auch auf dem Medion läuft.
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

# Sicherheitscheck: Stimmt der Projektname? Verhindert, dass die Struktur
# versehentlich in einem falschen Ordner angelegt wird.
EXPECTED_NAME = "_MyMSIT_AI-Agent_repair-playlist"
assert PROJECT_ROOT.name == EXPECTED_NAME, (
    f"Unerwarteter Projektordner: {PROJECT_ROOT}"
)

# --- Soll-Struktur laut Projektanweisungen v0.3 -------------------------------
ORDNER = [
    "config",
    "data/raw/crates",
    "data/raw/playlists",
    "data/raw/index",
    "data/interim",
    "data/processed",
    "data/gold",
    "notebooks",
    "src/playlist_agent/parsers",
    "src/playlist_agent/agent",
    "app",
    "tests",
    "reports/figures",
    "reports/tables",
    "reports/errors",
    "reports/run_reports",
    "docs",
    "presentation",
    "models",
    "logs",
    "tests/fixtures",
    "src/playlist_agent/agent/prompts",
]

# --- Anlegen und protokollieren ----------------------------------------------
protokoll = []
for rel in ORDNER:
    pfad = PROJECT_ROOT / rel
    status = "vorhanden" if pfad.exists() else "neu"
    pfad.mkdir(parents=True, exist_ok=True)
    (pfad / ".gitkeep").touch(exist_ok=True)
    protokoll.append({"ordner": rel, "status": status})

    
# --- Python-Paketstruktur: __init__.py erzeugen ------------------------------
# Warum: Erst __init__.py macht einen Ordner zu einem regulären Python-Paket.
#   Voraussetzung für "pip install -e ." und saubere Imports aus src/.
PAKETE = [
    "src/playlist_agent",
    "src/playlist_agent/parsers",
    "src/playlist_agent/agent",
]
for rel in PAKETE:
    init = PROJECT_ROOT / rel / "__init__.py"
    status = "vorhanden" if init.exists() else "neu"
    init.touch(exist_ok=True)
    protokoll.append({"ordner": f"{rel}/__init__.py", "status": status})


# --- Ergebnis als CSV speichern (Namensschema <NB>_<Zelle>_<slug>) -----------
csv_pfad = PROJECT_ROOT / "reports/tables/01_01_ordnerstruktur.csv"
with open(csv_pfad, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["ordner", "status"])
    writer.writeheader()
    writer.writerows(protokoll)

# --- Ausgabe -----------------------------------------------------------------
print(f"Python      : {sys.version.split()[0]}  ({sys.executable})")
print(f"Projektwurzel: {PROJECT_ROOT}\n")
for p in protokoll:
    print(f"  [{p['status']:9}] {p['ordner']}")
n_neu = sum(p["status"] == "neu" for p in protokoll)
print(f"\n{len(protokoll)} Ordner geprüft, davon {n_neu} neu angelegt.")
print(f"Protokoll gespeichert: {csv_pfad.relative_to(PROJECT_ROOT)}")

In [ ]:
# =============================================================================
# Zelle 02 – Projektdateien anlegen: .gitignore, pyproject.toml, LICENSE, README
# =============================================================================
# Zweck: Basisdateien reproduzierbar per Skript erzeugen.
# Sicherheit: Vorhandene Dateien werden NIE überschrieben (Status "vorhanden").
#   Änderungen danach direkt in der Datei, nicht durch erneutes Generieren.
# .gitignore-Logik: Inhalte privater Ordner werden ignoriert, die .gitkeep-
#   Platzhalter bleiben erhalten, damit die Ordnerstruktur im Repo sichtbar ist.
# =============================================================================

from datetime import date

COPYRIGHT_HOLDER = "AwaTekoete"  

GITIGNORE = """\
# =============================================================================
# .gitignore – public Repo: echte Daten, Musik und Secrets duerfen NIE hinein
# Prinzip: Ordner-Regeln (Kern) + Dateityp-Regeln (Sicherheitsnetz, falls
# echte Dateien versehentlich ausserhalb von data/ landen)
# =============================================================================

# --- 1. Private Ordner (Inhalt ignoriert, .gitkeep bleibt fuer die Struktur) ---
data/**
!data/**/
!data/**/.gitkeep
models/**
!models/.gitkeep
logs/**
!logs/.gitkeep
reports/run_reports/**
!reports/run_reports/.gitkeep
reports/errors/**
!reports/errors/.gitkeep

# --- 2. MLflow (lokales Tracking, ab Phase 4) ---
mlruns/
mlartifacts/
mlflow.db

# --- 3. Sicherheitsnetz: echte Playlists, Indizes, Musik, Modelle ueberall ---
*.crate
*.m3u
*.m3u8
*.wpl
*.fpl
*musik_index*
*.mp3
*.wav
*.aif
*.aiff
*.flac
*.m4a
*.joblib
*.pkl
# Ausnahme: kleine SYNTHETISCHE Testdaten (public-tauglich, s. Projektanweisung)
!tests/fixtures/**

# --- 4. Secrets (Vorlage .env.example bleibt im Repo) ---
.env
.env.*
!.env.example
*.key
*.pem

# --- 5. Python / Jupyter / Build ---
__pycache__/
*.py[cod]
*.egg-info/
build/
dist/
.ipynb_checkpoints/
.pytest_cache/
.venv/

# --- 6. Betriebssystem-Artefakte (u. a. AppleDouble vom Mac, s. EDA-Vorbefunde) ---
._*
.DS_Store
Thumbs.db
desktop.ini

# --- 7. Editor ---
.vscode/
"""

PYPROJECT = """\
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "playlist-agent"
version = "0.1.0"
description = "Agentic repair and sync of Serato/foobar2000 playlists - feasibility study (MSIT)"
readme = "README.md"
requires-python = ">=3.11"
license = "MIT"
authors = [{ name = "%s" }]
dependencies = []          # wird je Phase ergänzt (mutagen, rapidfuzz, ...)

[project.optional-dependencies]
dev = ["ipykernel", "nbstripout", "pytest"]

[tool.setuptools.packages.find]
where = ["src"]
""" % COPYRIGHT_HOLDER

LICENSE = f"""\
MIT License

Copyright (c) {date.today().year} {COPYRIGHT_HOLDER}

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
"""

README = """\
# Playlist Repair Agent

Feasibility study (MSIT, AI-Agent project): Where does deterministic engineering
suffice, and where do ML, fuzzy matching and LLM agents add measurable value
when repairing and synchronising Serato / foobar2000 playlists?

Status: Phase 0 – Setup

> No real music data is part of this repository. Demos use synthetic data only.
"""

DATEIEN = {
    ".gitignore": GITIGNORE,
    "pyproject.toml": PYPROJECT,
    "LICENSE": LICENSE,
    "README.md": README,
}

for name, inhalt in DATEIEN.items():
    pfad = PROJECT_ROOT / name
    if pfad.exists():
        print(f"  [vorhanden] {name}  (nicht überschrieben)")
    else:
        pfad.write_text(inhalt, encoding="utf-8")
        print(f"  [neu      ] {name}  ({len(inhalt.splitlines())} Zeilen)")

In [ ]:
# =============================================================================
# Zelle 03 – Projektkonfiguration config/config.yaml anlegen
# =============================================================================
# Zweck: Zentrale, versionierte Konfiguration. Code liest Werte von hier,
#   statt sie hart zu codieren (z. B. Zielordner je Laptop).
# Public Repo: KEINE Secrets, KEINE Seriennummer (-> .env, Schritt 0.6b).
# Herkunft jedes Werts ist im Kommentar belegt (Befund oder "offen").
# Sicherheit: Vorhandene Datei wird nicht überschrieben.
# =============================================================================

CONFIG_YAML = """\
# Projektkonfiguration Playlist Repair Agent
# Public: keine Secrets, keine Seriennummern (-> .env)
version: 1

stick:
  # Befund anker_info.txt (06.11.2026): Datentraegername "Lexar", FAT32
  volume_label: "Lexar"
  # Primaeres Erkennungsmerkmal (Projektanweisung Workflow 0): Ordner-Signatur.
  # Befund: alle 5 Ordner existieren auf dem Stick (Angabe Projektinhaber).
  signatur:
    - "Musik"
    - "Playlist"
    - "_ScratchLIVE_/Crates"
    - "Playlist_AsusExpertBook_foobar2000"
    - "Playlist_Medion_foobar2000"
  # Alle Signatur-Ordner muessen vorhanden sein. Lockerung nur mit Begruendung.
  signatur_min_treffer: 5

laptops:
  - name: "asus"
    hostname: "ASUSExpertbook"            # Befund: Ausgabe 'hostname'
    zielordner: "Playlist_AsusExpertBook_foobar2000"
    laufwerk_beobachtet: "D:"             # nur Doku, Logik nutzt Signatur
  - name: "medion"
    hostname: "LTMedionErik"              # Befund: Ausgabe 'hostname' Medion
    zielordner: "Playlist_Medion_foobar2000"
    laufwerk_beobachtet: "E:"             # Befund: Laurent_Garnier.m3u

quellen:
  crates: "_ScratchLIVE_/Crates"          # Serato, Wahrheitsquelle
  mac_playlists: "Playlist"               # Mac-M3U
  musik: "Musik"                          # Audio + .m3u/.wpl im Musikordner

index:
  # Befund Index v2: beobachtete Audio-Endungen. In EDA (Phase 1) verifizieren.
  audio_endungen: [".mp3", ".wav", ".aif"] # Videos (.m4v) bewusst ausgeschlossen
  # Befund Index v2: 151 AppleDouble-Dateien "._*", max. 4096 Byte
  appledouble_praefix: "._"
  appledouble_max_bytes: 4096

matching:
  # Schwellen werden in Phase 4 datenbasiert kalibriert (Precision = 100 %).
  # Bewusst leer: keine Annahmen.
  {}
"""

ziel = PROJECT_ROOT / "config" / "config.yaml"
if ziel.exists():
    print(f"[vorhanden] {ziel.relative_to(PROJECT_ROOT)}  (nicht überschrieben)")
else:
    ziel.write_text(CONFIG_YAML, encoding="utf-8")
    print(f"[neu      ] {ziel.relative_to(PROJECT_ROOT)}  "
          f"({len(CONFIG_YAML.splitlines())} Zeilen)")